In [1]:
base_url = "<TERMINOLOGY_API_URL>"  # replace with your app url

In [ ]:
%pip install requests-oauthlib

In [2]:
if not base_url.lower().startswith('https://'):
    import os
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "true"

In [3]:
import requests
import html
import re
import urllib.parse
from requests_oauthlib import OAuth2Session

## OpenId Authentication

#### Client_id and client_secret can be accessed by Admin. 
- Visit Terminology server API service Info Page. 
- Select API service Access on the left
- Copy client_id and client_secret

In [4]:
openid_conf = requests.get(f"{base_url}/.well-known/openid-configuration")
openid_conf_json = openid_conf.json()

In [5]:
client_id = "terminology_server" # replace with client_id
client_secret = "client-secret" # replace with client_secret
username = "username" # replace with username
password = "password" # replace with password

authorization_base_url = openid_conf_json["authorization_endpoint"]
token_url = openid_conf_json["token_endpoint"]
scope = ["email", "profile", "openid"]
redirect_uri = f"{base_url}/keycloak-callback"

session = OAuth2Session(client_id, scope=scope, redirect_uri=redirect_uri, pkce="S256")
authorization_url, state = session.authorization_url(authorization_base_url,access_type="offline")

In [6]:
print('Visit the link in browser, login with your credentials. After login, you will be redirected to a page that will not load. Copy this link')
authorization_url

Visit the link in browser, login with your credentials. After login, you will be redirected to a page that will not load. Copy this link


'http://3.91.27.92/auth/realms/terminology_server/protocol/openid-connect/auth?response_type=code&client_id=terminology_server&redirect_uri=http%3A%2F%2F3.91.27.92%2Fkeycloak-callback&scope=email+profile+openid&state=DrM5hLjEgJG0KwLpWAeR5SuauHG7BE&code_challenge=dXqRqZCPi2ZIYywU2fMy_O5ESXZ0pmUSzdYjXGs6g9I&code_challenge_method=S256&access_type=offline'

In [7]:
redirect = input("Past the copied redirect-link:")
tokens = session.fetch_token(token_url, authorization_response=redirect, include_client_id=True, client_secret=client_secret)

Past the copied redirect-link: http://3.91.27.92/keycloak-callback?state=DrM5hLjEgJG0KwLpWAeR5SuauHG7BE&session_state=324a1949-2c0a-406b-ae02-b512e7b6abf1&iss=http%3A%2F%2F3.91.27.92%2Fauth%2Frealms%2Fterminology_server&code=809a5f25-2add-4269-b8e2-17b73948f565.324a1949-2c0a-406b-ae02-b512e7b6abf1.c70a4756-e301-4eb1-95a8-939606de5d56


In [8]:
def map_one_code_to_another(source_codes, source_vocabulary=None, target_vocabulary=None):
    url = f"{base_url}/api/code_map"
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    payload = {
        "codes": source_codes,
        "source_vocabulary": source_vocabulary,
        "target_vocabulary": target_vocabulary
    }
    response = session.post(url, headers=headers, json=payload)
    return response.json()

## 1. Code to Code Mapping

This API provides mapping from one code system (source) to another code system (target).

In [9]:
source_codes = ["E11", "E14"]

map_one_code_to_another(source_codes=source_codes)

{'message': {'E11': [{'vocabulary': 'ICD10CM',
    'code': 'E11',
    'mappings': {'UMLS': ['C0011847',
      'C0011860',
      'C0851219',
      'C0854110',
      'C4290093']}},
   {'vocabulary': 'ICD10',
    'code': 'E11',
    'mappings': {'MEDDRA_PT': ['10067585'], 'MEDDRA_LLT': ['10067585']}}],
  'E14': [{'vocabulary': 'ICD10',
    'code': 'E14',
    'mappings': {'MEDDRA_PT': ['10012601'], 'MEDDRA_LLT': ['10012601']}}]}}

User can provide single or multiple codes as per their need.

It has feature to filter by source and/or target code system

In [10]:
source_codes = ["E11", "E14"]
source_vocabulary = "ICD10"
target_vocabulary = "MEDDRA_PT"

map_one_code_to_another(source_codes=source_codes, 
                                          source_vocabulary=source_vocabulary,
                                          target_vocabulary=target_vocabulary)

{'message': {'E11': [{'vocabulary': 'ICD10',
    'code': 'E11',
    'mappings': {'MEDDRA_PT': ['10067585']}}],
  'E14': [{'vocabulary': 'ICD10',
    'code': 'E14',
    'mappings': {'MEDDRA_PT': ['10012601']}}]}}

## 2. Term to Code Mapping (Search by Terms)

This API provides mapping of provided terms to respective codes and additionally provides other metadata details

In [11]:
def search(chunks, 
            search_type=None, 
            vocabulary_id=[], 
            validity=None,
            topk=20,
            standard_concept=[], 
            domain_id=[], 
            source=[], 
            concept_class_id=[], 
            filter_by_valueset=False,
            valueset_metadata_ids=[], 
            source_vocabulary=None,
            target_vocabulary=None,
           access_token=None):
    url = f"{base_url}/api/search"
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json"
    }
    payload = {
        "chunks": chunks,
        "search_type": search_type,
        "vocabulary_id": vocabulary_id,
        "validity": validity,
        "topk": topk,
        "standard_concept": standard_concept,
        "domain_id": domain_id,
        "source": source,
        "concept_class_id": concept_class_id,
        "filter_by_valueset": filter_by_valueset,
        "valueset_metadata_ids": valueset_metadata_ids,
        "source_vocabulary": source_vocabulary,
        "target_vocabulary": target_vocabulary
    }
    response = session.post(url, headers=headers, json=payload)
    return response.json()

### 2.1 Basic Search Example

Let's start with a simple search query. 
By default, search gives exact match as well as semantic search results.


In [12]:
search(chunks=["diabetes"])

{'detail': [{'type': 'too_short',
   'loc': ['body', 'vocabulary_id'],
   'msg': 'List should have at least 1 item after validation, not 0',
   'input': [],
   'ctx': {'field_type': 'List', 'min_length': 1, 'actual_length': 0}}]}

### 2.2 Exact match search

For searching results of exact match, we should pass parameter: 
- search_type = "exact"

Using additional filter parameters:
- topk -> to get top k results only
- validity -> setting it "true" to get only valid results
- domain_id -> list of domains. Eg: Condition, Observation, Measurement, Procedure, Meas Value
- vocabulary_id -> list of code systems. Eg: ICD10, ICD10CM, LOINC, SNOMED, MeSH


In [13]:
search(chunks=["viral hepatitis"], 
        topk=10, 
        search_type="exact",
        validity="true",
        domain_id=["Condition", "Observation"],
        vocabulary_id=["ICD10", "ICD9"]
        )

{'viral hepatitis': {'suggested_query': None, 'results': []}}

### 2.3 Semantic Search

For semantic search, we should pass the paramter:
- search_type = "semantic"

In [14]:
search(chunks=["viral hepatitis"], 
        topk=10, 
        search_type="semantic",
        validity="true",
        domain_id=["Condition", "Observation"],
        vocabulary_id=["ICD10", "ICD9"]
        )

{'viral hepatitis': {'suggested_query': None, 'results': []}}

### 2.4 Spell Checker feature in Search

User may pass a misspelled words in the keywords(chunks). API provides useful suggestion in the result section `suggested_query` when it finds such typo.

In [15]:
search(chunks=["liverr"])

{'detail': [{'type': 'too_short',
   'loc': ['body', 'vocabulary_id'],
   'msg': 'List should have at least 1 item after validation, not 0',
   'input': [],
   'ctx': {'field_type': 'List', 'min_length': 1, 'actual_length': 0}}]}

### 2.5 Search Integrated with Concept Maps

By passing parameters `source_vocabulary` and `target_vocabulary`, search can also provide code mappings from specified source to target (wherever possible).

In [16]:
search(chunks=["liver"], 
        topk=10,
        validity="true",
        domain_id=["Condition", "Observation"],
        vocabulary_id=["SNOMED"],
        source_vocabulary="SNOMED",
        target_vocabulary="UMLS"
        )

{'liver': {'suggested_query': None,
  'results': [{'chunk': 'liver',
    'document': 'liver tissue',
    'cmetadata': {'ID': 1277472,
     'source': 'Athena',
     'validity': True,
     'domain_id': 'Observation',
     'domain_name': 'Observation',
     'concept_code': '256887000',
     'vocabulary_id': 'SNOMED',
     'valid_end_date': '2099-12-31',
     'vocabulary_name': 'Systematic Nomenclature of Medicine - Clinical Terms (IHTSDO)',
     'chunk_concept_id': 4120372,
     'concept_class_id': 'Substance',
     'standard_concept': 'Standard',
     'valid_start_date': '2002-01-31',
     'domain_concept_id': 27,
     'concept_class_name': 'Substance',
     'vocabulary_concept_id': 44819097,
     'concept_class_concept_id': 44819050},
    'score': 0.8164,
    'value_set_records': [],
    'code_map_records': {'UMLS': [{'concept_code': 'C0736268',
       'concept_name': 'hepatic tissue'}]}},
   {'chunk': 'liver',
    'document': 'liver - meat',
    'cmetadata': {'ID': 1245086,
     'sourc

## 3. Code to Term Mapping (Search by Concept Code)

Instead of the concept term, We can search by code as well.
For this, we simply pass the code(s) via `chunks` parameter.

In [17]:
search(chunks=["B15-B19", "B18"], 
        topk=10, 
        search_type="exact",
        validity="true"
        )

{'detail': [{'type': 'too_short',
   'loc': ['body', 'vocabulary_id'],
   'msg': 'List should have at least 1 item after validation, not 0',
   'input': [],
   'ctx': {'field_type': 'List', 'min_length': 1, 'actual_length': 0}}]}

## 4. Document Search
Send documents to search for terms

### 4.1 Fast Search

In [20]:
import requests

url = f"{base_url}/api/document/search-task"

payload = {}
files=[
  ('files',('doc1.txt',open('/Users/neo/Downloads/doc1.txt','rb'),'text/plain')),
  ('files',('doc2.txt',open('/Users/neo/Downloads/doc2.txt','rb'),'text/plain'))
]
headers = {
  'Accept': 'application/json'
}

response = session.post(url, headers=headers, data=payload, files=files)

print(response.json())


{'task_id': '6d4235bc-9bc5-4ff0-8259-f1d1d805a647'}


In [21]:
task_id = response.json()['task_id']

In [22]:
doc_search_status = f"{base_url}/api/document/search-task/{task_id}"

doc_search_data = session.get(doc_search_status)
doc_search_data.json()

[{'document_name': 'doc1.txt',
  'content': 'Chief Complaint:         Patient presents with worsening shortness of breath for the past 3 days, associated with intermittent chest tightness and fatigue.          History of Present Illness:         The patient is a 64-year-old male with a significant medical history of long-standing hypertension, type 2 diabetes mellitus, hyperlipidemia, and a prior myocardial infarction in 2019. He reports that over the past 72 hours he has experienced progressively increasing shortness of breath, especially with exertion, and mild chest pressure radiating occasionally to the left shoulder. He denies syncope but reports two episodes of dizziness. No fever, chills, nausea, or abdominal pain. He mentions a chronic cough that worsened mildly this week with occasional sputum production.          Past Medical History:         - Hypertension diagnosed 12 years ago, currently on lisinopril.         - Type 2 diabetes mellitus diagnosed 8 years ago, on metformin 

## 4.2 Accurate Search

In [33]:
import requests
import json

url = f"{base_url}/api/document/accurate-search-task"

payload = {'is_accurate_search': 'true',
'entity_config': '{"entity_config":[{"entity":"lungs","vocabularies":["ICD10PCS"]}]}'}

files=[
  ('files',('doc1.txt',open('/Users/neo/Downloads/doc1.txt','rb'),'text/plain')),
  ('files',('doc2.txt',open('/Users/neo/Downloads/doc2.txt','rb'),'text/plain'))
]
headers = {
  'Accept': 'application/json'
}

response = session.post(url, headers=headers, data=payload, files=files)

print(response.json())
task_id = response.json()['task_id']

{'task_id': '3f409a91-8c79-45ee-a328-6eda573b1656'}


In [35]:
doc_search_status = f"{base_url}/api/document/search-task/{task_id}"

doc_search_data = session.get(doc_search_status)
doc_search_data.json()

[{'document_name': 'doc1.txt',
  'content': 'Chief Complaint:         Patient presents with worsening shortness of breath for the past 3 days, associated with intermittent chest tightness and fatigue.          History of Present Illness:         The patient is a 64-year-old male with a significant medical history of long-standing hypertension, type 2 diabetes mellitus, hyperlipidemia, and a prior myocardial infarction in 2019. He reports that over the past 72 hours he has experienced progressively increasing shortness of breath, especially with exertion, and mild chest pressure radiating occasionally to the left shoulder. He denies syncope but reports two episodes of dizziness. No fever, chills, nausea, or abdominal pain. He mentions a chronic cough that worsened mildly this week with occasional sputum production.          Past Medical History:         - Hypertension diagnosed 12 years ago, currently on lisinopril.         - Type 2 diabetes mellitus diagnosed 8 years ago, on metformin 